<a href="https://colab.research.google.com/github/Ramkanc/Capstone_IIITH/blob/Capstone1_MultiModel_V4/V1_MultiModal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import FLickr8k dataset from Kaggle hub

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adityajn105/flickr8k")

print("Path to dataset files:", path)

100%|██████████| 1.04G/1.04G [00:12<00:00, 86.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1


Copy the Kaggle data in to colab environment

In [2]:
import shutil
import os

In [3]:
# Create the dataset  directory if it doesn't exist
dataset_dir = '/content/dataset_dir'
os.makedirs(dataset_dir, exist_ok=True)

In [4]:
# Copy the dataset directory recursively
shutil.copytree(path, dataset_dir, dirs_exist_ok=True)
print(f"Dataset copied to {dataset_dir}")

Dataset copied to /content/dataset_dir


Find out content in the folder

In [5]:
found_directories = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
found_files =[f for f in os.listdir(dataset_dir) if os.path.isfile(os.path.join(dataset_dir, f))]
print (found_directories)
print (found_files)

['Images']
['captions.txt']


In [9]:
images_path = os.path.join(dataset_dir, found_directories[0])
print(images_path)

/content/dataset_dir/Images


In [11]:
captions_path = os.path.join(dataset_dir, found_files[0])
print(captions_path)

/content/dataset_dir/captions.txt


In [12]:
# Verify images directory
print(f"Total Images: {len(os.listdir(images_path))}")
print("Sample Images:", os.listdir(images_path)[:5])

Total Images: 8091
Sample Images: ['2704379125_9c35650d16.jpg', '2148695079_9ae6a9b1c7.jpg', '1713248099_d860df4e10.jpg', '3484649669_7bfe62080b.jpg', '3647693147_0d0434351b.jpg']


In [13]:
# Verify captions file
with open(captions_path, 'r') as file:
    captions_sample = [file.readline() for _ in range(5)]
print("Sample Captions:", captions_sample)

Sample Captions: ['image,caption\n', '1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set of stairs in an entry way .\n', '1000268201_693b08cb0e.jpg,A girl going into a wooden building .\n', '1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .\n', '1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playhouse .\n']


In [21]:
import pandas as pd

In [23]:
# Load captions into a DataFrame
captions = pd.read_csv(captions_path, sep=',', header=0, names=['image', 'caption'])

In [24]:
# Extract image names (removing the #index from filenames)
captions['image'] = captions['image'].str.split('.').str[0]

In [25]:
# Check the structure of the dataset
print(captions.head())
print(f"Total Caption Records: {len(captions)}")

                   image                                            caption
0  1000268201_693b08cb0e  A child in a pink dress is climbing up a set o...
1  1000268201_693b08cb0e              A girl going into a wooden building .
2  1000268201_693b08cb0e   A little girl climbing into a wooden playhouse .
3  1000268201_693b08cb0e  A little girl climbing the stairs to her playh...
4  1000268201_693b08cb0e  A little girl in a pink dress going into a woo...
Total Caption Records: 40455


In [26]:
# Group captions by image
grouped_captions = captions.groupby('image')['caption'].apply(list).reset_index()

In [27]:
# Inspect grouped captions
print(grouped_captions.head())
print(f"Total Unique Images: {len(grouped_captions)}")

                   image                                            caption
0  1000268201_693b08cb0e  [A child in a pink dress is climbing up a set ...
1  1001773457_577c3a7d70  [A black dog and a spotted dog are fighting, A...
2  1002674143_1b742ab4b8  [A little girl covered in paint sits in front ...
3  1003163366_44323f5815  [A man lays on a bench while his dog sits by h...
4  1007129816_e794419615  [A man in an orange hat starring at something ...
Total Unique Images: 8091


In [28]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [34]:
# Download NLTK data (if not already available)
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [35]:
# Stopwords for English
stop_words = set(stopwords.words('english'))

In [36]:
# Function to preprocess text
def preprocess_caption(caption):
    # Convert to lowercase
    caption = caption.lower()
    # Remove non-alphanumeric characters
    caption = re.sub(r'[^a-z0-9\s]', '', caption)
    # Tokenize
    tokens = word_tokenize(caption)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

In [37]:
# Apply preprocessing to all captions
grouped_captions['processed_captions'] = grouped_captions['caption'].apply(lambda captions: [preprocess_caption(caption) for caption in captions])

In [38]:
# View processed captions
print(grouped_captions.head())

                   image                                            caption  \
0  1000268201_693b08cb0e  [A child in a pink dress is climbing up a set ...   
1  1001773457_577c3a7d70  [A black dog and a spotted dog are fighting, A...   
2  1002674143_1b742ab4b8  [A little girl covered in paint sits in front ...   
3  1003163366_44323f5815  [A man lays on a bench while his dog sits by h...   
4  1007129816_e794419615  [A man in an orange hat starring at something ...   

                                  processed_captions  
0  [child pink dress climbing set stairs entry wa...  
1  [black dog spotted dog fighting, black dog tri...  
2  [little girl covered paint sits front painted ...  
3  [man lays bench dog sits, man lays bench white...  
4  [man orange hat starring something, man wears ...  


In [39]:
from sklearn.model_selection import train_test_split

# Split into train, val, test (70%, 15%, 15%)
train_data, test_data = train_test_split(grouped_captions, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(test_data, test_size=0.5, random_state=42)

print(f"Training Samples: {len(train_data)}")
print(f"Validation Samples: {len(val_data)}")
print(f"Testing Samples: {len(test_data)}")

Training Samples: 5663
Validation Samples: 1214
Testing Samples: 1214


In [40]:
# Save datasets to CSV
train_data.to_csv("flickr8k_train.csv", index=False)
val_data.to_csv("flickr8k_val.csv", index=False)
test_data.to_csv("flickr8k_test.csv", index=False)

print("Data saved successfully!")

Data saved successfully!


In [41]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
from PIL import Image


In [42]:
class FlickrDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.data = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = f"{self.image_dir}/{row['image']}"
        captions = row['processed_captions']

        # Load and transform the image
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, captions


In [43]:
# Image transformations
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_df = pd.read_csv("flickr8k_train.csv")
val_df = pd.read_csv("flickr8k_val.csv")
test_df = pd.read_csv("flickr8k_test.csv")

train_dataset = FlickrDataset(train_df, "flickr8k/Flickr_Data/Images", transform=image_transforms)
val_dataset = FlickrDataset(val_df, "flickr8k/Flickr_Data/Images", transform=image_transforms)
test_dataset = FlickrDataset(test_df, "flickr8k/Flickr_Data/Images", transform=image_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [44]:
class ImageEncoder(nn.Module):
    def __init__(self, embed_size):
        super(ImageEncoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad = False  # Freeze ResNet parameters
        resnet.fc = nn.Linear(resnet.fc.in_features, embed_size)  # Replace FC layer
        self.resnet = resnet

    def forward(self, images):
        features = self.resnet(images)
        return features


In [45]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(TextEncoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, embed_size)

    def forward(self, captions):
        embeddings = self.embedding(captions)
        _, (hidden, _) = self.lstm(embeddings)
        features = self.fc(hidden[-1])  # Use the last hidden state
        return features


In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, image_embed_size, text_vocab_size, text_embed_size, text_hidden_size, text_num_layers):
        super(MultimodalModel, self).__init__()
        self.image_encoder = ImageEncoder(image_embed_size)
        self.text_encoder = TextEncoder(text_vocab_size, text_embed_size, text_hidden_size, text_num_layers)

    def forward(self, images, captions):
        image_features = self.image_encoder(images)
        text_features = self.text_encoder(captions)
        return image_features, text_features


In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, image_features, text_features):
        similarity = torch.cosine_similarity(image_features, text_features)
        loss = 1 - similarity.mean()
        return loss

# Instantiate the model, loss function, and optimizer
model = MultimodalModel(
    image_embed_size=256,
    text_vocab_size=5000,
    text_embed_size=256,
    text_hidden_size=512,
    text_num_layers=2
).to("cuda")

criterion = ContrastiveLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for images, captions in train_loader:
            images = images.to("cuda")
            captions = captions.to("cuda")

            # Forward pass
            image_features, text_features = model(images, captions)
            loss = criterion(image_features, text_features)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer, num_epochs=10)


In [ ]:
def generate_caption(image, model, tokenizer):
    model.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to("cuda")
        image_features = model.image_encoder(image)

        # Decode text features into words (implement beam search for better results)
        # This part depends on your tokenizer and vocabulary implementation.
